# Кандидатогенерация услуг Авито, Recall@50

Нужны папка dataset с тремя parquet-файлами и видеокарта (я запускал на RTX 3060 с 12 ГБ).

Идея решения такая. Для каждого запроса по всем объявлениям корпуса считается несколько оценок: BM25, косинус эмбеддингов дообученного e5-small, вероятность локации объявления при данной локации поиска (её выучиваем из train), расстояние до центра локации поиска и категории, которые выбирали по похожим запросам. Из разных комбинаций этих оценок берём по 200 лучших объявлений, склеиваем в пул (в среднем около 470 штук на запрос) и отдаём LightGBM-ранкеру, который оставляет 50.

О повторяемости. Веса дообученного энкодера лежат в artifacts/e5_ft. С ними ноутбук выдаёт файл, идентичный отправленному на stepik. Если папки с весами нет, ноутбук сам дообучит энкодер за 25 минут, но обучение на GPU не повторяется бит в бит, так что файл получится очень похожим, а не идентичным. Всё остальное (индексы, эмбеддинги, пулы, ранкер) считается детерминированно, полный прогон занимает около часа.

In [1]:
import os, sys, re, math, time, pickle, hashlib
from collections import Counter
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn.functional as F
import lightgbm as lgb
from pymorphy3 import MorphAnalyzer
from transformers import AutoModel, AutoTokenizer

pd.set_option("display.width", 200)
# все пути считаем от текущей папки: dataset с данными, work_nb для кэша, artifacts с весами энкодера
ROOT = Path.cwd()
DATA = ROOT / "dataset"
WORK = ROOT / "work_nb"; WORK.mkdir(exist_ok=True)
WEIGHTS = ROOT / "artifacts" / "e5_ft"
OUT = ROOT / "answer.csv"
assert torch.cuda.is_available(), "нужна видеокарта с CUDA"

# тяжёлые шаги (индексы, эмбеддинги, пулы) считаем один раз и кладём на диск,
# чтобы повторный запуск ячейки не ждал заново
def cached(path, make, kind):
    path = Path(path)
    if path.exists():
        return {"npy": np.load, "pkl": lambda p: pickle.load(open(p, "rb")), "parquet": pd.read_parquet}[kind](path)
    obj = make()
    {"npy": lambda: np.save(path, obj), "pkl": lambda: pickle.dump(obj, open(path, "wb"), protocol=4),
     "parquet": lambda: obj.to_parquet(path)}[kind]()
    return pd.read_parquet(path) if kind == "parquet" else obj


## 1. Данные

Читаем train (497 тысяч пар запрос-объявление), 2452 запроса бенчмарка и корпус из 189 тысяч объявлений. Ниже функции, которые читают данные и готовят тексты. Из параметров объявления вырезается адрес, для смысла он не нужен, и достаются значения "Вид услуги" и "Тип услуги", чтобы сравнивать их с фильтрами запроса.

In [2]:
# один seed на все случайные операции
SEED = 42
# колонки, по которым запрос считается одним и тем же, и колонки объявления
QUERY_COLS = ["search_query", "search_location_id", "search_is_delivery_search",
              "search_infm_params_text", "search_category"]
ITEM_COLS = ["item_id", "item_title_raw", "item_description_raw", "item_infm_params_text",
             "item_category_id", "item_microcat_id", "item_price", "item_rating",
             "item_rating_reviews_count", "item_location_id", "item_latitude",
             "item_longitude", "item_is_phone_hidden", "item_is_message_forbidden"]


# в parquet цена и координаты лежат как Decimal, приводим к float32
def _num(df, cols):
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("float32")
    return df


# читаем три файла; пропуски в текстовых полях заменяем пустой строкой
def load_raw():
    tr = pd.read_parquet(DATA / "train.parquet")
    bq = pd.read_parquet(DATA / "benchmark_queries.parquet")
    bi = pd.read_parquet(DATA / "benchmark_items.parquet")
    for df in (tr, bi):
        _num(df, ["item_price", "item_latitude", "item_longitude"])
        df["item_infm_params_text"] = df["item_infm_params_text"].fillna("")
        df["item_description_raw"] = df["item_description_raw"].fillna("")
        df["item_title_raw"] = df["item_title_raw"].fillna("")
    for df in (tr, bq):
        df["search_infm_params_text"] = df["search_infm_params_text"].fillna("")
    return tr, bq, bi


# токен это последовательность латиницы, кириллицы или цифр
_TOK = re.compile(r"[a-zа-я0-9]+")


def normalize(s: str) -> str:
    return s.lower().replace("ё", "е")


def tokenize(s: str):
    return _TOK.findall(normalize(s))


# параметры объявления это длинная строка вида 'Вид услуги ... Место оказания услуг ... Тип услуги ...'.
# Три регулярки ниже вырезают адрес и достают вид и тип услуги. Список ключей, на которых
# значение обрывается, собран по частым ключам в корпусе
_ADDR = re.compile(r"Место оказания услуг.*?(?= (?:Тип |Направление|Опыт|Как вы|Работаете|Гарантия|Где |Ваши |Кто |"
                   r"Берёте|Марка|Услуги|Формат|Чем |Рабочие|Время |График|Предоплата|Название)|$)")
_VID = re.compile(r"Вид услуги (.+?)(?= (?:Место оказания|Тип |Направление|Опыт|Как вы|Работаете|Гарантия|Где |"
                  r"Ваши |Кто |Берёте|Марка|Услуги|Формат|Чем |Рабочие|Онлайн|Рейтинг)|$)")
_TIP = re.compile(r"Тип услуги (.+?)(?= (?:Место оказания|Тип |Направление|Опыт|Как вы|Работаете|Гарантия|Где |"
                  r"Ваши |Кто |Берёте|Марка|Услуги|Формат|Чем |Рабочие|Онлайн|Рейтинг|Вид услуги)|$)")


# адрес для смысла бесполезен, в текст для энкодера он не попадает
def clean_params(s: str, max_len: int = 220) -> str:
    return _ADDR.sub("", s)[:max_len].strip()


def extract_vid(s: str) -> str:
    m = _VID.search(s)
    return normalize(m.group(1)).strip() if m else ""


def extract_tip(s: str) -> str:
    m = _TIP.search(s)
    return normalize(m.group(1)).strip() if m else ""


# то, что видит энкодер: заголовок, параметры и начало описания
def doc_text(title: str, params: str, desc: str, desc_len: int = 400) -> str:
    d = " ".join(desc.split())[:desc_len]
    return f"{title} | {clean_params(params)} | {d}"


# фильтры поиска дописываем к тексту запроса
def query_text(q: str, infm: str) -> str:
    return f"{q} | {infm}" if infm else q


In [3]:
# беглый взгляд на данные, из него следуют решения ниже
tr, bq, bi = load_raw()
print("train:", tr.shape, "| запросов бенчмарка:", len(bq), "| корпус:", len(bi))
print("доля объявлений train, лежащих в корпусе: %.3f" % tr.item_id.drop_duplicates().isin(bi.item_id).mean())
print("совпадение локации поиска и объявления в train: %.3f" % (tr.search_location_id == tr.item_location_id).mean())
print("запросов бенчмарка, в локации которых нет объявлений корпуса: %.3f" % (~bq.search_location_id.isin(set(bi.item_location_id))).mean())
print("тексты запросов бенчмарка, встречавшиеся в train: %.3f" % bq.search_query.isin(set(tr.search_query)).mean())
print("запросы бенчмарка без фильтров: %.3f (в train %.3f); без категории (search_category=0): %.3f" % (
    (bq.search_infm_params_text == "").mean(), (tr.search_infm_params_text == "").mean(), (bq.search_category == 0).mean()))


train: (497673, 19) | запросов бенчмарка: 2452 | корпус: 189212
доля объявлений train, лежащих в корпусе: 0.053
совпадение локации поиска и объявления в train: 0.831
запросов бенчмарка, в локации которых нет объявлений корпуса: 0.174
тексты запросов бенчмарка, встречавшиеся в train: 0.370
запросы бенчмарка без фильтров: 0.631 (в train 0.330); без категории (search_category=0): 0.091


Из этого следует три вещи. Объявления train почти не встречаются в корпусе (около 5%), так что искать по train нельзя, он нужен только для обучения. Локация решает многое: в 83% случаев выбранное объявление из той же локации, что и поиск, а для 17% запросов бенчмарка в их локации нет ни одного объявления корпуса, и клики уходят в соседние локации. Эту связь придётся выучить из train. И 63% текстов запросов бенчмарка в train не встречались, значит одной памяти о запросах мало, нужна семантика.

## 2. Как проверяем качество

Разметки бенчмарка нет, поэтому качество проверяем на hold-out из train и стараемся сделать его похожим на бенчмарк.

Первая версия hold-out оказалась слишком лёгкой: на ней вышло 0.953, а stepik показал 0.887. Причина в том, что в корпусе бенчмарка вокруг запроса лежат близкие по смыслу объявления (похоже, клики других пользователей по тем же запросам), а у запроса из hold-out таких соседей в корпусе не было. Поэтому схема теперь такая.

Берём 9000 запросов, по одному на уникальный текст и с тем же распределением по локациям, что у бенчмарка, и убираем их объявления из обучения. Остальные объявления train делим пополам. Половина A идёт на обучение энкодера и статистик. Половину G энкодер не видит никогда, и 120 тысяч её объявлений мы кладём в корпус валидации как конкурентов, так же как в бенчмарке лежат объявления, которых нет в train. Ещё 15 тысяч запросов берём из G, они нужны только для обучения ранкера. Оцениваем на первых 3000 hold-out запросах.

Порядок строк в корпусе важен: новые объявления всегда дописываются в конец, от этого зависит кэш эмбеддингов.

In [4]:
# делит train на три части: обучающую, hold-out запросы и объявления-конкуренты
# (зачем это нужно, написано выше)
def make_split(tr: pd.DataFrame, bq: pd.DataFrame, n_val: int = 3000, seed: int = SEED, g_frac: float = 0.5):
    rng = np.random.RandomState(seed)
    tr = tr.copy()
    tr["qkey"] = pd.factorize(tr[QUERY_COLS].astype(str).agg("|".join, axis=1))[0]
    # по одному запросу на текст, как в бенчмарке, где все тексты разные
    firsts = tr.drop_duplicates("qkey")
    firsts = firsts.groupby("search_query", sort=False).sample(n=1, random_state=seed)
    # вес запроса: доля его локации в бенчмарке, делённая на число запросов из этой локации.
    # Так распределение локаций у hold-out повторяет бенчмарк
    share = bq["search_location_id"].value_counts(normalize=True)
    per_loc = firsts["search_location_id"].value_counts()
    w = firsts["search_location_id"].map(share).fillna(0) / firsts["search_location_id"].map(per_loc)
    w = w.to_numpy()
    p = w / w.sum()
    pick = rng.choice(len(firsts), size=n_val, replace=False, p=p)
    val_q = set(firsts.iloc[pick]["qkey"])
    is_val = tr["qkey"].isin(val_q)
    val = tr[is_val].reset_index(drop=True)
    val_items = set(val["item_id"])
    # объявления hold-out запросов выкидываем из обучения целиком
    rest = tr[~is_val & ~tr["item_id"].isin(val_items)]
    # оставшиеся объявления делим пополам: A идёт в обучение, G энкодер не увидит,
    # и оно пойдёт в корпус валидации как конкуренты
    rng2 = np.random.RandomState(seed + 1)
    uniq = rest["item_id"].unique()
    in_g = set(uniq[rng2.rand(len(uniq)) < g_frac])
    is_g = rest["item_id"].isin(in_g)
    return rest[~is_g].reset_index(drop=True), val, rest[is_g].reset_index(drop=True)


In [5]:
# сколько берём hold-out запросов, дополнительных запросов для ранкера и объявлений-конкурентов
N_HOLD = 9000
N_EXTRA_Q = 15_000
N_DISTRACT = 120_000


# контейнер: корпус, запросы и (для валидации) разметка
class World:
    def __init__(self, name, corpus, queries, rel=None):
        self.name = name
        self.corpus = corpus.reset_index(drop=True)
        self.queries = queries.reset_index(drop=True)
        self.item_index = {k: i for i, k in enumerate(self.corpus.item_id)}
        self.rel = rel


# собирает обе выборки: val с разметкой и test, который нужно предсказать
def build(n_hold=N_HOLD):
    tr, bq, bi = load_raw()
    train, hold, dist = make_split(tr, bq, n_val=n_hold)
    # корпус валидации: корпус бенчмарка, релевантные объявления hold-out и часть объявлений G
    known = set(bi.item_id)
    extra = hold.drop_duplicates("item_id")[ITEM_COLS]
    extra = extra[~extra.item_id.isin(known)]
    conc = dist.drop_duplicates("item_id")[ITEM_COLS]
    conc = conc[~conc.item_id.isin(known | set(extra.item_id))]
    conc = conc.sample(n=min(len(conc), N_DISTRACT), random_state=SEED)
    val_corpus = pd.concat([bi[ITEM_COLS], extra, conc], ignore_index=True)
    qcols = [c for c in hold.columns if c.startswith("search_")] + ["qkey"]
    vq = hold.drop_duplicates("qkey")[qcols].reset_index(drop=True)
    n_hold_q = len(vq)
    # дополнительные запросы для ранкера берём из G, чтобы энкодер не видел ни запрос, ни его объявления
    g_only = dist[~dist.qkey.isin(set(train.qkey))]
    ex_keys = set()
    if N_EXTRA_Q:
        fx = g_only.drop_duplicates("qkey").groupby("search_query", sort=False).sample(n=1, random_state=SEED)
        share = bq["search_location_id"].value_counts(normalize=True)
        w = (fx["search_location_id"].map(share).fillna(0) / fx["search_location_id"].map(fx["search_location_id"].value_counts())).to_numpy()
        pick = np.random.RandomState(SEED + 2).choice(len(fx), size=N_EXTRA_Q, replace=False, p=w / w.sum())
        ex_keys = set(fx.iloc[pick]["qkey"])
    ex_rows = g_only[g_only.qkey.isin(ex_keys)]
    ex_items = ex_rows.drop_duplicates("item_id")[ITEM_COLS]
    ex_items = ex_items[~ex_items.item_id.isin(set(val_corpus.item_id))]
    # новые объявления дописываем в конец, чтобы номера прежних не сдвинулись
    # (от них зависит кэш эмбеддингов)
    val_corpus = pd.concat([val_corpus, ex_items], ignore_index=True)
    vq = pd.concat([vq, ex_rows.drop_duplicates("qkey")[qcols]], ignore_index=True)
    hold = pd.concat([hold, ex_rows], ignore_index=True)
    val = World("val", val_corpus, vq)
    val.rel = _rel(hold, vq, val)
    # строки G нужны только для статистики по локациям. Строки дополнительных запросов убираем,
    # иначе статистика подсмотрит их собственные клики
    val.stats_rows = dist[~dist.qkey.isin(ex_keys)][["search_location_id", "item_location_id", "item_latitude", "item_longitude"]].copy()
    # оцениваем только на первых 3000 hold-out запросах, остальные пойдут в обучение ранкера
    val.queries["is_eval"] = np.arange(len(vq)) < 3000
    test = World("test", bi[ITEM_COLS], bq)
    return train, val, test


# релевантные объявления берём как множество, повторные клики по одной паре не считаем
def _rel(hold, vq, world):
    g = hold.groupby("qkey").item_id.apply(list)
    return {i: sorted({world.item_index[x] for x in g[k]}) for i, k in enumerate(vq.qkey)}


In [6]:
# обучающая часть, выборка для валидации и выборка бенчмарка
train, val, test = build()
# размеры корпуса и числа запросов val до добавления запросов для ранкера.
# Эмбеддинги ниже считаются двумя порциями по этой границе
N_BASE, Q_BASE = 317_848, 9_000
assert len(val.corpus) >= N_BASE and len(val.queries) >= Q_BASE
print("обучающих пар (часть A):", len(train))
print("val: объявлений", len(val.corpus), "запросов", len(val.queries), "(оценочных %d)" % val.queries.is_eval.sum())
print("test: объявлений", len(test.corpus), "запросов", len(test.queries))
print("доля hold-out запросов с текстом, встречавшимся в обучении: %.3f" % val.queries.iloc[:Q_BASE].search_query.isin(set(train.search_query)).mean())


обучающих пар (часть A): 239963
val: объявлений 321215 запросов 24000 (оценочных 3000)
test: объявлений 189212 запросов 2452
доля hold-out запросов с текстом, встречавшимся в обучении: 0.278


## 3. Лексический канал

Лемматизация через pymorphy3 и BM25 отдельно по заголовку, параметрам и описанию. BM25 считается умножением разреженных матриц, это быстрее, чем перебирать документы в цикле. Кроме того, для каждой пары считаем, какая доля лемм запроса встретилась в заголовке и во всём тексте.

In [7]:
# лемматизация нужна, чтобы 'видеодомофон' и 'видеодомофонов' считались одним словом
_morph = MorphAnalyzer()


# уникальных слов на порядок меньше, чем токенов, поэтому лемму кэшируем
@lru_cache(maxsize=2_000_000)
def lemma(word: str) -> str:
    if word.isdigit() or not word[0].isalpha():
        return word
    return _morph.parse(word)[0].normal_form.replace("ё", "е")


def lemmas(text: str):
    return [lemma(w) for w in tokenize(text)]


# словарь лемм: каждой лемме соответствует свой столбец матрицы
class Vocab:

    def __init__(self):
        self.t2i = {}

    def add(self, toks):
        for t in toks:
            if t not in self.t2i:
                self.t2i[t] = len(self.t2i)

    def ids(self, toks, grow=False):
        if grow:
            self.add(toks)
        return [self.t2i[t] for t in toks if t in self.t2i]


# разреженная матрица частот: строки документы, столбцы леммы
def count_matrix(token_lists, vocab: Vocab, grow: bool):
    indptr, indices, data = [0], [], []
    for toks in token_lists:
        c = Counter(vocab.ids(toks, grow=grow))
        indices.extend(c.keys())
        data.extend(c.values())
        indptr.append(len(indices))
    return sp.csr_matrix((np.asarray(data, dtype=np.float32), indices, indptr),
                         shape=(len(token_lists), max(len(vocab.t2i), 1)))


# BM25 считаем отдельно по трём полям
FIELDS = {"title": "item_title_raw", "params": "item_infm_params_text", "desc": "item_description_raw"}


# индекс по корпусу: BM25 для каждого поля плюс частоты для признака покрытия
class LexIndex:

    def __init__(self, corpus):
        self.vocab = Vocab()
        toks = {f: [lemmas(x) for x in corpus[col]] for f, col in FIELDS.items()}
        for f in toks:
            for t in toks[f]:
                self.vocab.add(t)
        self.bm = {f: BM25Field().fit(count_matrix(toks[f], self.vocab, grow=False)) for f in toks}
        # частоты по заголовку и по всему тексту пригодятся для признака покрытия запроса
        self.title_tf = count_matrix(toks["title"], self.vocab, grow=False)
        self.any_tf = sum(count_matrix(toks[f], self.vocab, grow=False) for f in toks)

    # запрос превращаем в бинарный вектор лемм
    def query_matrix(self, texts):
        Q = count_matrix([lemmas(x) for x in texts], self.vocab, grow=False)
        Q.data[:] = 1
        return Q

    def field_scores(self, Q):
        return {f: self.bm[f].score(Q).toarray() for f in self.bm}

    # какая доля лемм запроса нашлась в заголовке и во всём тексте объявления
    def coverage(self, Q):
        n = np.maximum(np.asarray(Q.sum(1)).ravel(), 1)
        out = {}
        for name, tf in (("cov_title", self.title_tf), ("cov_any", self.any_tf)):
            B = tf.copy(); B.data[:] = 1
            out[name] = (Q @ B.T).toarray() / n[:, None]
        return out


# BM25 одного поля
class BM25Field:

    def __init__(self, k1=1.2, b=0.75):
        self.k1, self.b = k1, b

    def fit(self, tf: sp.csr_matrix):
        n_docs = tf.shape[0]
        dl = np.asarray(tf.sum(1)).ravel()
        avgdl = max(dl.mean(), 1e-6)
        df = np.bincount(tf.indices, minlength=tf.shape[1])
        self.idf = np.log(1 + (n_docs - df + 0.5) / (df + 0.5)).astype(np.float32)
        # вес терма в документе: насыщение по частоте с поправкой на длину документа.
        # Матрицу сразу храним транспонированной, чтобы запросы умножались на неё одной операцией
        tf = tf.tocoo()
        denom = tf.data + self.k1 * (1 - self.b + self.b * dl[tf.row] / avgdl)
        w = tf.data * (self.k1 + 1) / denom
        self.W_T = sp.csr_matrix((w.astype(np.float32), (tf.row, tf.col)), shape=tf.shape).T.tocsr()
        return self

    # запрос (леммы, умноженные на idf) умножаем на матрицу весов
    def score(self, q_bin: sp.csr_matrix):
        q = q_bin.multiply(self.idf[None, :]).tocsr()
        return q @ self.W_T


In [8]:
# индексы по корпусу val и по корпусу бенчмарка, лемматизация идёт несколько минут
t0 = time.time()
lex_val = cached(WORK / "lex_val.pkl", lambda: LexIndex(val.corpus), "pkl")
lex_test = cached(WORK / "lex_test.pkl", lambda: LexIndex(test.corpus), "pkl")
print("индексы готовы, %.0f c, словарь val: %d лемм" % (time.time() - t0, len(lex_val.vocab.t2i)))


индексы готовы, 1 c, словарь val: 550652 лемм


## 4. Энкодер

Берём multilingual-e5-small и дообучаем на парах "запрос и выбранное объявление" из части A. Потеря контрастивная: для запроса положительный пример это его объявление, отрицательные это остальные объявления батча. Батчи собраны из четырёх микрокатегорий подряд, чтобы среди негативов были объявления на близкую тему. Пары с одинаковым текстом запроса или с тем же объявлением в одном батче маскируем, иначе они превращаются в ложные негативы (один и тот же запрос встречается в train до тысячи раз). Параметры: батч 128, две эпохи, lr 4e-5, температура 0.05, bf16. Модель побольше (e5-base) в 12 ГБ не поместилась.

In [9]:
# e5-small; запрос обрезаем до 32 токенов, документ до 192
BASE_MODEL = "intfloat/multilingual-e5-small"
Q_LEN, D_LEN = 32, 192


class Encoder:
    def __init__(self, path=BASE_MODEL, device="cuda"):
        self.tok = AutoTokenizer.from_pretrained(path)
        self.model = AutoModel.from_pretrained(path).to(device)
        self.device = device

    # среднее по токенам с учётом маски и L2-нормировка, как принято у e5
    def embed(self, texts, prefix, max_len, grad=False):
        b = self.tok([prefix + t for t in texts], padding=True, truncation=True,
                     max_length=max_len, return_tensors="pt").to(self.device)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            out = self.model(**b).last_hidden_state
        m = b["attention_mask"].unsqueeze(-1).to(out.dtype)
        e = (out * m).sum(1) / m.sum(1)
        return F.normalize(e.float(), dim=-1)

    @torch.no_grad()
    # тексты сортируем по длине, чтобы в батчах было меньше паддинга
    def encode(self, texts, prefix, max_len, bs=256):
        self.model.eval()
        order = np.argsort([len(t) for t in texts])
        out = np.zeros((len(texts), self.model.config.hidden_size), dtype=np.float16)
        for s in range(0, len(texts), bs):
            idx = order[s:s + bs]
            out[idx] = self.embed([texts[i] for i in idx], prefix, max_len).cpu().numpy().astype(np.float16)
        return out


# собирает батчи из нескольких микрокатегорий подряд. Так среди негативов
# много объявлений на близкую тему, а не только случайные
def make_batches(microcat, batch_size, buckets_per_batch, rng):
    n = len(microcat)
    perm = rng.permutation(n)
    order = perm[np.argsort(microcat[perm], kind="stable")]
    mc_order = rng.permutation(np.unique(microcat))
    rank = {m: i for i, m in enumerate(mc_order)}
    order = order[np.argsort([rank[m] for m in microcat[order]], kind="stable")]
    chunk = batch_size // buckets_per_batch
    pieces = [order[i:i + chunk] for i in range(0, n, chunk)]
    rng.shuffle(pieces)
    batches = []
    for i in range(0, len(pieces) - buckets_per_batch + 1, buckets_per_batch):
        batches.append(np.concatenate(pieces[i:i + buckets_per_batch]))
    return batches


# контрастивное обучение: положительный пример запроса это его объявление,
# отрицательные это остальные объявления батча
def train_encoder(enc: Encoder, q_texts, d_texts, microcat, q_ids, i_ids, *, epochs=1, batch_size=128,
          lr=2e-5, temperature=0.05, max_steps=None, seed=42, log_every=100):
    rng = np.random.RandomState(seed)
    torch.manual_seed(seed)
    q_ids, i_ids, microcat = map(np.asarray, (q_ids, i_ids, microcat))
    steps_per_epoch = len(q_texts) // batch_size
    total = min(max_steps or 10**9, steps_per_epoch * epochs)
    opt = torch.optim.AdamW(enc.model.parameters(), lr=lr, weight_decay=0.01)
    # разогрев 200 шагов, дальше косинусный спад
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, s / 200) * max(0.0, 0.5 * (1 + math.cos(math.pi * s / total))))
    enc.model.train()
    step, t0, run = 0, time.time(), 0.0
    for ep in range(epochs):
        for b in make_batches(microcat, batch_size, 4, rng):
            if step >= total:
                return
            qe = enc.embed([q_texts[i] for i in b], "query: ", Q_LEN)
            de = enc.embed([d_texts[i] for i in b], "passage: ", D_LEN)
            logits = qe @ de.T / temperature
            # один текст запроса встречается много раз, а у запроса бывает несколько объявлений.
            # Такие пары в батче нельзя считать негативами, поэтому маскируем их
            same = torch.from_numpy((q_ids[b][:, None] == q_ids[b][None, :]) |
                                    (i_ids[b][:, None] == i_ids[b][None, :])).to(logits.device)
            eye = torch.eye(len(b), dtype=torch.bool, device=logits.device)
            logits = logits.masked_fill(same & ~eye, -1e4)
            labels = torch.arange(len(b), device=logits.device)
            # потеря в обе стороны: запрос находит объявление и объявление находит запрос
            loss = 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(enc.model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
            run = 0.98 * run + 0.02 * loss.item() if step else loss.item()
            step += 1
            if step % log_every == 0:
                print(f"step {step}/{total} loss {run:.4f} {time.time()-t0:.0f}s", flush=True)


In [10]:
# если готовых весов нет, обучаем энкодер сами
if not (WEIGHTS / "config.json").exists():
    print("весов нет, обучаем энкодер с нуля")
    enc = Encoder()
    # обучающие пары: текст запроса и текст выбранного объявления
    q_texts = [query_text(q, i) for q, i in zip(train.search_query, train.search_infm_params_text)]
    d_texts = [doc_text(t, p, d) for t, p, d in zip(train.item_title_raw, train.item_infm_params_text, train.item_description_raw)]
    # две эпохи по 120 тысяч пар, около 25 минут на RTX 3060
    train_encoder(enc, q_texts, d_texts, train.item_microcat_id.values, pd.factorize(train.search_query)[0],
                  pd.factorize(train.item_id)[0], epochs=2, batch_size=128, lr=4e-5)
    WEIGHTS.mkdir(parents=True, exist_ok=True)
    enc.model.save_pretrained(WEIGHTS); enc.tok.save_pretrained(WEIGHTS)
    del enc; torch.cuda.empty_cache()
else:
    print("используем готовые веса", WEIGHTS)


используем готовые веса E:\Avito NLP fin\artifacts\e5_ft


Дальше кодируем всё, что понадобится. Границы порций (N_BASE и Q_BASE) те же, что были при расчёте отправленного файла.

In [11]:
# кодируем всё, что понадобится дальше: корпус val, заголовки, запросы val и test
# и уникальные запросы обучающей части (для коллаборативного канала)
t0 = time.time()
c, q = val.corpus, val.queries
docs = lambda df: [doc_text(t, p, d) for t, p, d in zip(df.item_title_raw, df.item_infm_params_text, df.item_description_raw)]
qtexts = lambda df: [query_text(a, b) for a, b in zip(df.search_query, df.search_infm_params_text)]
uq = train.drop_duplicates("qkey")[["qkey", "search_query", "search_infm_params_text"]].reset_index(drop=True)
# энкодер загружаем лениво, только если какого-то кэша не хватает
_enc = {}
def enc_obj():
    if "e" not in _enc:
        _enc["e"] = Encoder(str(WEIGHTS))
    return _enc["e"]

# корпус кодируем двумя порциями (первые N_BASE строк и хвост), как при расчёте отправленного файла.
# Иначе батчи сложатся по-другому и эмбеддинги чуть изменятся
emb_c = cached(WORK / "emb_corpus.npy", lambda: np.concatenate([
    enc_obj().encode(docs(c.iloc[:N_BASE]), "passage: ", 192), enc_obj().encode(docs(c.iloc[N_BASE:]), "passage: ", 192)]), "npy")
emb_t = cached(WORK / "emb_title.npy", lambda: np.concatenate([
    enc_obj().encode(list(c.item_title_raw.iloc[:N_BASE]), "passage: ", 48), enc_obj().encode(list(c.item_title_raw.iloc[N_BASE:]), "passage: ", 48)]), "npy")
emb_q_val = cached(WORK / "emb_q_val.npy", lambda: np.concatenate([
    enc_obj().encode(qtexts(q.iloc[:Q_BASE]), "query: ", 32), enc_obj().encode(qtexts(q.iloc[Q_BASE:]), "query: ", 32)]), "npy")
emb_q_test = cached(WORK / "emb_q_test.npy", lambda: enc_obj().encode(qtexts(test.queries), "query: ", 32), "npy")
emb_q_train = cached(WORK / "emb_q_train.npy", lambda: enc_obj().encode(qtexts(uq), "query: ", 32), "npy")
_enc.clear(); torch.cuda.empty_cache()
print("эмбеддинги готовы, %.0f c:" % (time.time() - t0), emb_c.shape, emb_t.shape, emb_q_val.shape, emb_q_test.shape, emb_q_train.shape)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

эмбеддинги готовы, 395 c: (321215, 384) (321215, 384) (24000, 384) (2452, 384) (186759, 384)


## 5. География и коллаборативный канал

География. Из train выучиваем матрицу "локация поиска, локация объявления" (с какой вероятностью объявление из данной локации выбирают при поиске из другой) и центр каждой локации поиска, это медиана координат выбранных по ней объявлений. Для пары запрос-объявление получаем вероятность локации и расстояние до центра в километрах. Статистики считаются по всем строкам train, кроме hold-out.

Коллаборативный канал. Для запроса находим 30 ближайших запросов из обучающей части (по эмбеддингам), усредняем по ним распределения микрокатегории, вида и типа услуги выбранных объявлений и получаем для каждого объявления вероятность его категории. Это спасает там, где BM25 промахивается по словам: запрос "скупка телевизоров" и объявление "выкуп техники".

In [12]:
# из train выучиваем, в каких локациях лежат объявления, которые выбирают при поиске из данной локации
class LocPrior:
    def __init__(self, train: pd.DataFrame, extra_locs=()):
        # частоты пар (локация поиска, локация объявления) и условная вероятность локации объявления
        tm = train.groupby(["search_location_id", "item_location_id"]).size().rename("n").reset_index()
        tm["p"] = tm.n / tm.groupby("search_location_id").n.transform("sum")
        s_locs = sorted(set(tm.search_location_id))
        i_locs = sorted(set(tm.item_location_id) | set(extra_locs))
        self.s_idx = {l: i for i, l in enumerate(s_locs)}
        self.i_idx = {l: i for i, l in enumerate(i_locs)}
        self.T = sp.csr_matrix(
            (tm.p.values.astype(np.float32),
             ([self.s_idx[a] for a in tm.search_location_id], [self.i_idx[b] for b in tm.item_location_id])),
            shape=(len(s_locs), len(i_locs)))
        # общее распределение локаций объявлений; в признаках не участвует
        g = train.item_location_id.value_counts(normalize=True)
        self.global_p = np.zeros(len(i_locs), dtype=np.float32)
        for l, v in g.items():
            if l in self.i_idx:
                self.global_p[self.i_idx[l]] = v

        # центр локации поиска: медиана координат объявлений, выбранных по ней.
        # Расстояние до центра работает и там, где точная пара локаций в train не встречалась
        cent = train.dropna(subset=["item_latitude", "item_longitude"]).groupby(
            "search_location_id")[["item_latitude", "item_longitude"]].median()
        self.cent = {l: (np.radians(a), np.radians(b)) for l, a, b in
                     zip(cent.index, cent.item_latitude, cent.item_longitude)}

    # расстояние по формуле гаверсинуса; если центра локации нет, подставляем 3000 км
    def dist_km(self, search_locations, lat_rad, lon_rad, unknown_km=3000.0):
        la0 = np.array([self.cent.get(l, (np.nan, np.nan))[0] for l in search_locations])[:, None]
        lo0 = np.array([self.cent.get(l, (np.nan, np.nan))[1] for l in search_locations])[:, None]
        a = np.sin((lat_rad[None] - la0) / 2) ** 2 + np.cos(la0) * np.cos(lat_rad[None]) * \
            np.sin((lon_rad[None] - lo0) / 2) ** 2
        km = 6371.0 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
        return np.where(np.isnan(km), unknown_km, km).astype(np.float32)

    def item_loc_idx(self, item_locations):
        return np.array([self.i_idx.get(l, -1) for l in item_locations])

    # матрица (запросы x объявления): вероятность локации каждого объявления при локации поиска запроса
    def probs(self, search_locations, item_loc_idx):
        rows = np.array([self.s_idx.get(l, -1) for l in search_locations])
        known = rows >= 0
        out = np.zeros((len(rows), len(item_loc_idx)), dtype=np.float32)
        if known.any():
            dense = self.T[rows[known]].toarray()
            ok = item_loc_idx >= 0
            m = np.zeros((int(known.sum()), len(item_loc_idx)), dtype=np.float32)
            m[:, ok] = dense[:, item_loc_idx[ok]]
            out[known] = m
        return out


In [13]:
# для каждого запроса train: доля выбранных объявлений по каждому значению категории
class CatDist:

    def __init__(self, qkey_idx, values, n_queries, C, device="cuda"):
        M = np.zeros((n_queries, C), dtype=np.float32)
        np.add.at(M, (qkey_idx, values), 1.0)
        M /= np.maximum(M.sum(1, keepdims=True), 1)
        self.M = torch.from_numpy(M).to(device).half()
        self.C = C


# по эмбеддингу запроса берём 30 ближайших запросов train и усредняем их распределения категорий
# (микрокатегория, вид, тип услуги) с весами softmax по сходству
class Collab:
    def __init__(self, train: pd.DataFrame, train_queries: pd.DataFrame, emb_train_q, cat_defs,
                 device="cuda", k=30, temp=0.05, max_cats=300):
        self.device, self.k, self.temp = device, k, temp
        self.E = torch.from_numpy(emb_train_q).to(device)
        pos = pd.Series(np.arange(len(train_queries)), index=train_queries.qkey.values)
        qidx = pos.loc[train.qkey.values].values
        self.enc = {}
        for name, (tr_vals, corp_vals) in cat_defs.items():
            codes, _ = pd.factorize(pd.concat([pd.Series(tr_vals), pd.Series(corp_vals)], ignore_index=True))
            n_tr = len(tr_vals)
            # оставляем max_cats самых частых значений категории, остальные сваливаем в одну группу.
            # Без этого матрица для типов услуг не помещалась в память
            freq = np.bincount(codes[:n_tr], minlength=codes.max() + 1)
            top = np.argsort(-freq)[:max_cats]
            remap = np.full(codes.max() + 1, len(top), dtype=np.int64)
            remap[top] = np.arange(len(top))
            codes = remap[codes]
            cd = CatDist(qidx, codes[:n_tr], len(train_queries), len(top) + 1, device)
            self.enc[name] = (cd, codes[n_tr:])

    @torch.no_grad()
    # для каждого объявления получаем вероятность его категории при данном запросе
    def aggregate(self, q_emb, exclude_self=None):
        q = torch.from_numpy(q_emb).to(self.device)
        sims = q @ self.E.T
        top_s, top_i = sims.float().topk(self.k, dim=1)
        w = torch.softmax(top_s / self.temp, dim=1)
        out = {"nn_sim": top_s[:, 0].cpu().numpy()}
        for name, (cd, item_codes) in self.enc.items():
            agg = torch.einsum("qk,qkc->qc", w, cd.M[top_i].float())
            out[name] = (agg, item_codes)
        return out


In [14]:
# числовые признаки объявлений считаем один раз на весь корпус
class ItemTable:

    def __init__(self, corpus: pd.DataFrame):
        c = corpus
        self.n = len(c)
        self.microcat = c.item_microcat_id.values
        self.rating = c.item_rating.fillna(-1).values.astype(np.float32)
        self.reviews = c.item_rating_reviews_count.fillna(0).values.astype(np.float32)
        self.price = np.log1p(c.item_price.clip(lower=0).fillna(0).values.astype(np.float32))
        self.phone_hidden = c.item_is_phone_hidden.values.astype(np.float32)
        self.msg_forbidden = c.item_is_message_forbidden.values.astype(np.float32)
        self.title_len = c.item_title_raw.str.len().values.astype(np.float32)
        self.desc_len = c.item_description_raw.str.len().values.astype(np.float32)
        self.params_len = c.item_infm_params_text.str.len().values.astype(np.float32)
        self.vid = pd.factorize(c.item_infm_params_text.map(extract_vid))[0]
        self.vid_str = c.item_infm_params_text.map(extract_vid).values
        self.tip_str = c.item_infm_params_text.map(extract_tip).values
        self.lat = c.item_latitude.fillna(0).values.astype(np.float32)
        self.lon = c.item_longitude.fillna(0).values.astype(np.float32)


# что запрос просит через фильтры: вид услуги, тип услуги, рейтинг
def query_filter_features(queries: pd.DataFrame, items: ItemTable):
    q_vid = queries.search_infm_params_text.map(extract_vid).values
    q_tip = queries.search_infm_params_text.map(extract_tip).values
    q_rating = queries.search_infm_params_text.str.contains("Рейтинг пользователя").values
    return q_vid, q_tip, q_rating


# 1 если объявление подходит под фильтр, 0 если нет, NaN если фильтр в запросе не задан
def filter_match(q_vid, q_tip, q_rating, items: ItemTable, idx):
    n = items.n
    vid = np.full((len(idx), n), np.nan, dtype=np.float32)
    tip = np.full((len(idx), n), np.nan, dtype=np.float32)
    rat = np.full((len(idx), n), np.nan, dtype=np.float32)
    for r, qi in enumerate(idx):
        if q_vid[qi]:
            vid[r] = (items.vid_str == q_vid[qi]).astype(np.float32)
        if q_tip[qi]:
            tip[r] = (items.tip_str == q_tip[qi]).astype(np.float32)
        if q_rating[qi]:
            rat[r] = (items.rating >= 4).astype(np.float32)
    return vid, tip, rat


## 6. Пул кандидатов и признаки

Для пачки из 100 запросов считаем матрицы оценок по всему корпусу и строим из них семь комбинаций. Из каждой берём 200 лучших объявлений, к ним добавляем по 30 лучших по чисто плотному и чисто лексическому каналу. Объявления не из категории 114 исключаем: во всём train выбранные объявления относятся к ней. Для каждой пары собирается 61 признак: оценки каналов, приоры, география, совпадение фильтров, рейтинг и прочие атрибуты объявления, а также ранги и отставание от лучшего кандидата внутри пула.

In [15]:
# запросы обрабатываем пачками по 100, иначе матрицы запросы x корпус не помещаются в память
CHUNK = 100


def _topk(H, k):
    return np.argpartition(-H, k, axis=1)[:, :k]


# главная функция. Для пачки запросов считает оценки всех каналов по всему корпусу,
# строит из них несколько комбинаций, берёт top-K каждой в общий пул кандидатов
# и собирает признаки пар для ранкера
def build_pool(world, lex, lp, items: ItemTable, emb_c, emb_t, emb_q, collab, cfg, only=None, log=print):
    qs = world.queries
    n = len(qs)
    idx_all = np.arange(n) if only is None else np.asarray(only)
    Qbin = lex.query_matrix(qs.search_query)
    ili = lp.item_loc_idx(world.corpus.item_location_id)
    # размер локации считаем по корпусу бенчмарка, чтобы в val и в test признак был в одном масштабе
    ref = ili[:cfg.get("n_ref", len(ili))]
    item_loc_size = np.bincount(ref[ref >= 0], minlength=len(lp.i_idx))[np.maximum(ili, 0)].astype(np.float32)
    same_loc_ids = world.corpus.item_location_id.values
    q_vid, q_tip, q_rat = query_filter_features(qs, items)
    q_len = qs.search_query.str.split().str.len().values.astype(np.float32)
    E_c = torch.from_numpy(emb_c).cuda()
    E_t = torch.from_numpy(emb_t).cuda()
    # в train все выбранные объявления из категории 114, остальные исключаем (штраф в оценках)
    not_service = (world.corpus.item_category_id.values != 114).astype(np.float32) * 100.0
    lat_rad = np.radians(world.corpus.item_latitude.values.astype(np.float64))
    lon_rad = np.radians(world.corpus.item_longitude.values.astype(np.float64))
    rows = []
    K = cfg["K"]
    for s in range(0, len(idx_all), CHUNK):
        idx = idx_all[s:s + CHUNK]
        fs = lex.field_scores(Qbin[idx])
        cov = lex.coverage(Qbin[idx])
        # лексическая оценка: заголовок весит втрое больше остальных полей
        lexc = 3 * fs["title"] + fs["params"] + fs["desc"]
        P = lp.probs(qs.search_location_id.values[idx], ili)
        qe = torch.from_numpy(emb_q[idx]).cuda()
        # плотное сходство запроса с полным текстом объявления и только с заголовком
        D = (qe @ E_c.T).float().cpu().numpy()
        Dt = (qe @ E_t.T).float().cpu().numpy()
        # приоры категорий по похожим запросам train
        agg = collab.aggregate(emb_q[idx])
        prior = {}
        for name in ("mc", "vid", "tip"):
            a, codes = agg[name]
            prior[name] = a[:, torch.from_numpy(codes).cuda()].cpu().numpy()
        # география: логарифм вероятности локации и логарифм расстояния
        logP = np.log(P + 3e-3)
        logD = np.log1p(lp.dist_km(qs.search_location_id.values[idx], lat_rad, lon_rad))
        # семь комбинаций оценок. H1: BM25 с приором локации; H2: плотное сходство с приором локации;
        # H3: то же плюс категорийный приор; H4: локация и расстояние; H5: только расстояние, без id локаций
        H1 = lexc * (P + 1e-3) ** 0.3
        H2 = D + cfg["a_loc"] * logP
        H3 = D + cfg["a_loc"] * logP + cfg["b_mc"] * np.log(prior["mc"] + 1e-2)
        H4 = D + 0.03 * logP - 0.02 * logD
        H5 = D - 0.04 * logD + cfg["b_mc"] * np.log(prior["mc"] + 1e-2)
        # H6: сходство и с полным текстом, и с заголовком
        H6 = 0.5 * (D + Dt) + 0.03 * logP - 0.02 * logD + cfg["b_mc"] * np.log(prior["mc"] + 1e-2)
        # H7: расширение запроса. Добавляем к запросу средний эмбеддинг трёх лучших по H3 объявлений
        # и ищем заново
        t3 = np.argpartition(-(H3 - not_service[None, :]), 3, axis=1)[:, :3]
        prf = qe + cfg.get("prf_w", 0.7) * E_c[torch.from_numpy(t3).cuda()].float().mean(1)
        prf = torch.nn.functional.normalize(prf, dim=-1).half()
        Dp = (prf @ E_c.T).float().cpu().numpy()
        H7 = Dp + 0.03 * logP - 0.02 * logD + cfg["b_mc"] * np.log(prior["mc"] + 1e-2)
        # штрафуем не-услуги во всех комбинациях
        for H in (H1, H2, H3, H4, H5, H6, H7):
            H -= not_service[None, :]
        Hs = {"h1": H1, "h2": H2, "h3": H3, "h4": H4, "h5": H5, "h6": H6, "h7": H7}
        vid_m, tip_m, rat_m = filter_match(q_vid, q_tip, q_rat, items, idx)
        for r, qi in enumerate(idx):
            # пул: top-K по каждой комбинации плюс top-30 по чисто плотному и по чисто лексическому каналу
            cand = np.unique(np.concatenate([_topk(H[r:r + 1], K)[0] for H in Hs.values()] +
                                            [_topk(D[r:r + 1], 30)[0], _topk(lexc[r:r + 1], 30)[0]]))
            # признаки пары запрос-объявление
            f = {
                "qidx": np.full(len(cand), qi), "item": cand,
                "bm_title": fs["title"][r, cand], "bm_params": fs["params"][r, cand], "bm_desc": fs["desc"][r, cand],
                "cov_title": cov["cov_title"][r, cand], "cov_any": cov["cov_any"][r, cand], "lexc": lexc[r, cand],
                "dense": D[r, cand], "p_loc": P[r, cand], "log_p_loc": logP[r, cand],
                "same_loc": (same_loc_ids[cand] == qs.search_location_id.values[qi]).astype(np.float32),
                "loc_size": item_loc_size[cand],
                "mc_prior": prior["mc"][r, cand], "vid_prior": prior["vid"][r, cand], "tip_prior": prior["tip"][r, cand],
                "vid_match": vid_m[r, cand], "tip_match": tip_m[r, cand], "rat_match": rat_m[r, cand],
                "nn_sim": np.full(len(cand), agg["nn_sim"][r]), "q_len": np.full(len(cand), q_len[qi]),
                "rating": items.rating[cand], "reviews": items.reviews[cand], "price": items.price[cand],
                "phone_hidden": items.phone_hidden[cand], "msg_forbidden": items.msg_forbidden[cand],
                "title_len": items.title_len[cand], "desc_len": items.desc_len[cand], "params_len": items.params_len[cand],
                "h1": H1[r, cand], "h2": H2[r, cand], "h3": H3[r, cand], "h4": H4[r, cand], "h5": H5[r, cand],
                "h6": H6[r, cand], "h7": H7[r, cand], "dense_title": Dt[r, cand], "dense_prf": Dp[r, cand],
                "log_dist": logD[r, cand],
            }
            # метка для обучения ранкера, есть только у валидации
            if world.rel is not None:
                f["label"] = np.isin(cand, world.rel[qi]).astype(np.int8)
            rows.append(pd.DataFrame(f))
        if (s // CHUNK) % 5 == 0:
            log(f"{world.name}: {s + len(idx)}/{len(idx_all)} queries")
    df = pd.concat(rows, ignore_index=True)
    # для главных оценок добавляем ранг внутри пула запроса и отставание от лучшего кандидата
    g = df.groupby("qidx")
    for c in ("lexc", "dense", "dense_title", "dense_prf", "h1", "h2", "h3", "h4", "h5", "h6", "h7", "mc_prior"):
        df[f"{c}_rank"] = g[c].rank(ascending=False, method="first").astype(np.float32)
        df[f"{c}_gap"] = df[c] - g[c].transform("max")
    return df


In [16]:
# гиперпараметры пула: веса локации и категорийного приора в комбинациях, число кандидатов K.
# n_ref задаёт корпус, по которому считается размер локации
t0 = time.time()
cfg = {"a_loc": 0.045, "b_mc": 0.02, "K": 200, "n_ref": len(test.corpus)}
# статистики локаций считаем по всем строкам train, кроме hold-out (обучающая часть плюс строки G)
cols = ["search_location_id", "item_location_id", "item_latitude", "item_longitude"]
lp = LocPrior(pd.concat([train[cols], val.stats_rows[cols]], ignore_index=True), extra_locs=val.corpus.item_location_id.unique())

# для выборки (val или test) собираем таблицы объявлений, коллаборативный канал и считаем пул
def make_pool(world, lex, emb_q):
    items = ItemTable(world.corpus)
    n = len(world.corpus)
    tvid = train.item_infm_params_text.map(extract_vid).values
    ttip = train.item_infm_params_text.map(extract_tip).values
    cat_defs = {"mc": (train.item_microcat_id.values, items.microcat), "vid": (tvid, items.vid_str), "tip": (ttip, items.tip_str)}
    collab = Collab(train, uq, emb_q_train, cat_defs)
    return build_pool(world, lex, lp, items, emb_c[:n], emb_t[:n], emb_q, collab, cfg,
                      log=lambda m: print(m, "%.0f c" % (time.time() - t0), flush=True))

# самый долгий шаг, около получаса
pool_val = cached(WORK / "pool_val.parquet", lambda: make_pool(val, lex_val, emb_q_val), "parquet")
pool_test = cached(WORK / "pool_test.parquet", lambda: make_pool(test, lex_test, emb_q_test), "parquet")
# первая проверка: какую долю релевантных объявлений пул вообще содержит
ev_idx = np.where(val.queries.is_eval)[0]
got = pool_val.groupby("qidx").item.apply(set)
print("средний размер пула: val %.0f, test %.0f" % (pool_val.groupby("qidx").size().mean(), pool_test.groupby("qidx").size().mean()))
print("покрытие пулом релевантных объявлений на оценочных запросах: %.4f" % np.mean([len(set(val.rel[a]) & got[a]) / len(val.rel[a]) for a in ev_idx]))


val: 100/24000 queries 13 c
val: 600/24000 queries 49 c
val: 1100/24000 queries 87 c
val: 1600/24000 queries 123 c
val: 2100/24000 queries 159 c
val: 2600/24000 queries 195 c
val: 3100/24000 queries 230 c
val: 3600/24000 queries 267 c
val: 4100/24000 queries 302 c
val: 4600/24000 queries 339 c
val: 5100/24000 queries 375 c
val: 5600/24000 queries 410 c
val: 6100/24000 queries 446 c
val: 6600/24000 queries 481 c
val: 7100/24000 queries 516 c
val: 7600/24000 queries 555 c
val: 8100/24000 queries 593 c
val: 8600/24000 queries 634 c
val: 9100/24000 queries 670 c
val: 9600/24000 queries 706 c
val: 10100/24000 queries 747 c
val: 10600/24000 queries 783 c
val: 11100/24000 queries 818 c
val: 11600/24000 queries 854 c
val: 12100/24000 queries 890 c
val: 12600/24000 queries 926 c
val: 13100/24000 queries 962 c
val: 13600/24000 queries 996 c
val: 14100/24000 queries 1032 c
val: 14600/24000 queries 1067 c
val: 15100/24000 queries 1103 c
val: 15600/24000 queries 1138 c
val: 16100/24000 queries 1172

## 7. Ранкер и ответ

LightGBM с целью lambdarank (NDCG@50). Ранкер учится на запросах, не входящих в оценочные 3000, число деревьев определяется ранней остановкой на оценочных. Параметры (31 лист, l2 = 10, feature_fraction 0.6, lr 0.03) подобраны на hold-out, более сложные модели переобучались. Затем итоговая модель обучается на всех 24 тысячах запросов и применяется к запросам бенчмарка, для каждого берём 50 объявлений с лучшим скором.

In [17]:
# параметры подобраны на hold-out. Модель посложнее переобучалась, запросов для обучения не так много
NON_FEATURES = {"qidx", "item", "label"}
PARAMS = dict(objective="lambdarank", metric="ndcg", eval_at=[50], learning_rate=0.03, num_leaves=31,
              min_data_in_leaf=200, lambda_l2=10.0, feature_fraction=0.6, bagging_fraction=0.8, bagging_freq=1,
              lambdarank_truncation_level=100, verbose=-1, seed=42, num_threads=8)


def feature_cols(df):
    return [c for c in df.columns if c not in NON_FEATURES]


def _group_sizes(df):
    return df.groupby("qidx", sort=False).size().values


# lambdarank, группа это запрос; если передана валидация, останавливаемся по ней
def fit(train_df: pd.DataFrame, valid_df: pd.DataFrame = None, rounds=600, params=None):
    p = {**PARAMS, **(params or {})}
    cols = feature_cols(train_df)
    dtr = lgb.Dataset(train_df[cols], train_df.label, group=_group_sizes(train_df))
    valids, cbs = [], [lgb.log_evaluation(100)]
    if valid_df is not None:
        valids = [lgb.Dataset(valid_df[cols], valid_df.label, group=_group_sizes(valid_df))]
        cbs.append(lgb.early_stopping(100))
    model = lgb.train(p, dtr, num_boost_round=rounds, valid_sets=valids, callbacks=cbs)
    model.cols = cols
    return model


# для каждого запроса оставляем k объявлений с лучшим скором ранкера
def top50(model, df: pd.DataFrame, k=50):
    df = df.assign(score=model.predict(df[model.cols], num_iteration=model.best_iteration or None))
    df = df.sort_values(["qidx", "score"], ascending=[True, False])
    return df.groupby("qidx", sort=False).head(k)[["qidx", "item", "score"] + (["label"] if "label" in df else [])]


# Recall@k как в задании: доля релевантных объявлений запроса, попавших в топ, среднее по запросам
def recall_at_k(pool: pd.DataFrame, world, qidx_subset, k=50):
    got = pool.groupby("qidx").item.apply(lambda s: set(s.head(k)))
    r = []
    for qi in qidx_subset:
        R = set(world.rel[qi])
        r.append(len(R & got.get(qi, set())) / len(R))
    return float(np.mean(r))


In [18]:
# ранкер учится на всех запросах, кроме 3000 оценочных
ev_q = set(ev_idx)
is_ev = pool_val.qidx.isin(ev_q)
tr_df, ev_df = pool_val[~is_ev].reset_index(drop=True), pool_val[is_ev].reset_index(drop=True)
print("пар для обучения ранкера: %d (запросов %d), пар для оценки: %d" % (len(tr_df), tr_df.qidx.nunique(), len(ev_df)))
# число деревьев выбираем по ранней остановке на оценочных запросах
model = fit(tr_df, ev_df, rounds=1500)
best = model.best_iteration
top = top50(model, ev_df)
print("Recall@50 на hold-out (3000 запросов): %.4f, итераций %d" % (recall_at_k(top, val, sorted(ev_q)), best))

# итоговую модель учим на всех 24000 запросах с найденным числом деревьев
final = fit(pool_val, None, rounds=best)
top_t = top50(final, pool_test)
# пишем ответ: query_id и 50 item_id через пробел
ids = test.corpus.item_id.values
ans = top_t.groupby("qidx").item.apply(lambda s: " ".join(ids[s.values]))
out = pd.DataFrame({"query_id": test.queries.query_id.values, "answer": [ans.get(i, "") for i in range(len(test.queries))]})
out.to_csv(OUT, index=False)
print("записан", OUT, out.shape)


пар для обучения ранкера: 9897996 (запросов 21000), пар для оценки: 1413901
Training until validation scores don't improve for 100 rounds
[100]	valid_0's ndcg@50: 0.613125
[200]	valid_0's ndcg@50: 0.619968
[300]	valid_0's ndcg@50: 0.618352
Early stopping, best iteration is:
[214]	valid_0's ndcg@50: 0.621083
Recall@50 на hold-out (3000 запросов): 0.9241, итераций 214
записан E:\Avito NLP fin\answer.csv (2452, 2)


## 8. Проверка файла

Проверяем формат по требованиям задания: две колонки, по строке на каждый query_id, не больше 50 идентификаторов в строке, без повторов, все идентификаторы из корпуса.

In [19]:
# проверяем формат так же, как это будет делать stepik
a = pd.read_csv(OUT, dtype=str, keep_default_na=False)
assert list(a.columns) == ["query_id", "answer"]
assert len(a) == len(bq) and a.query_id.is_unique and set(a.query_id) == set(bq.query_id) and a.query_id.str.len().eq(16).all()
corpus_ids = set(bi.item_id)
for s in a.answer:
    x = s.split(" ")
    assert 1 <= len(x) <= 50 and len(set(x)) == len(x)
    assert all(re.fullmatch(r"[0-9a-f]{16}", i) and i in corpus_ids for i in x)
print("формат корректен:", len(a), "строк, по", a.answer.str.split().str.len().min(), "-", a.answer.str.split().str.len().max(), "идентификаторов")


формат корректен: 2452 строк, по 50 - 50 идентификаторов


## 9. Что даёт каждый канал и где мы ошибаемся

Сначала Recall@50 на тех же 3000 оценочных запросах для отдельных каналов и для итоговой схемы, потом разбор потерянных пар по сегментам.

In [20]:
# сколько даёт каждый канал по отдельности на тех же оценочных запросах
E = torch.from_numpy(emb_c).cuda(); Eq = emb_q_val
ili = lp.item_loc_idx(val.corpus.item_location_id)
lat = np.radians(val.corpus.item_latitude.values.astype(np.float64)); lon = np.radians(val.corpus.item_longitude.values.astype(np.float64))
Qb = lex_val.query_matrix(val.queries.search_query)
not_service = (val.corpus.item_category_id.values != 114) * 100.0

# top-50 по заданной оценке и recall относительно разметки
def recall_of(fn, bs=100):
    hits = []
    for s in range(0, len(ev_idx), bs):
        idx = ev_idx[s:s + bs]
        H = fn(idx) - not_service[None, :]
        t = np.argpartition(-H, 50, axis=1)[:, :50]
        hits += [len(set(val.rel[a]) & set(t[r])) / len(val.rel[a]) for r, a in enumerate(idx)]
    return np.mean(hits)

def lexc(i):
    f = lex_val.field_scores(Qb[i]); return 3 * f["title"] + f["params"] + f["desc"]
dense = lambda i: (torch.from_numpy(Eq[i]).cuda() @ E.T).float().cpu().numpy()
P = lambda i: lp.probs(val.queries.search_location_id.values[i], ili)
LD = lambda i: np.log1p(lp.dist_km(val.queries.search_location_id.values[i], lat, lon))
abl = {
    "BM25, без географии": recall_of(lexc),
    "BM25 с приором локации": recall_of(lambda i: lexc(i) * (P(i) + 1e-3) ** 0.3),
    "плотный e5-small, без географии": recall_of(dense),
    "плотный с приором локации": recall_of(lambda i: dense(i) + 0.045 * np.log(P(i) + 3e-3)),
    "плотный с локацией и расстоянием": recall_of(lambda i: dense(i) + 0.03 * np.log(P(i) + 3e-3) - 0.02 * LD(i)),
    "пул кандидатов (потолок для ранкера)": np.mean([len(set(val.rel[a]) & got[a]) / len(val.rel[a]) for a in ev_idx]),
    "пул и ранкер (итог)": recall_at_k(top, val, sorted(ev_q)),
}
del E; torch.cuda.empty_cache()
pd.Series(abl, name="Recall@50").round(4).to_frame()


,Recall@50
"BM25, без географии",0.3393
BM25 с приором локации,0.7737
"плотный e5-small, без географии",0.3843
плотный с приором локации,0.8262
плотный с локацией и расстоянием,0.8352
пул кандидатов (потолок для ранкера),0.9685
пул и ранкер (итог),0.9241


In [21]:
# какие признаки итоговая модель использует сильнее всего
imp = pd.Series(final.feature_importance("gain"), index=final.cols).sort_values(ascending=False)
print("Наиболее важные признаки итоговой модели (gain):"); print(imp.head(10).round(0).astype(int).to_string())


Наиболее важные признаки итоговой модели (gain):
h7_gap           1556024
h7_rank           682528
h3_gap            652479
h4_gap            590962
cov_any           283060
h1_rank           174568
p_loc             122780
log_dist          121187
mc_prior_rank      75783
loc_size           64101


In [22]:
# разбор ошибок: для каждой релевантной пары смотрим, попала ли она в пул и в top-50,
# и считаем долю попаданий по сегментам
c, q = val.corpus, val.queries
size = c.iloc[:len(test.corpus)].item_location_id.value_counts()
pool_sets = pool_val.groupby("qidx").item.apply(set); g50 = top.groupby("qidx").item.apply(set)
seen = set(train.search_query)
rows = []
for a in ev_idx:
    for it in val.rel[a]:
        rows.append(dict(in_pool=it in pool_sets[a], in_top=it in g50[a],
                         same_loc=c.item_location_id.iloc[it] == q.search_location_id.iloc[a],
                         loc_n=size.get(q.search_location_id.iloc[a], 0), empty_filter=q.search_infm_params_text.iloc[a] == "",
                         text_seen=q.search_query.iloc[a] in seen))
d = pd.DataFrame(rows)
d["объявлений корпуса в локации поиска"] = pd.cut(d.loc_n, [-1, 0, 100, 1000, 5000, 30000], labels=["0", "1-100", "101-1000", "1001-5000", "более 5000"])
print("релевантных пар: %d, в top-50: %.4f, в пуле: %.4f" % (len(d), d.in_top.mean(), d.in_pool.mean()))
print("потеряно: не попало в пул %.4f, в пуле но ниже 50-го места %.4f" % ((~d.in_pool).mean(), (d.in_pool & ~d.in_top).mean()))
for col in ["same_loc", "empty_filter", "text_seen", "объявлений корпуса в локации поиска"]:
    print(); print(d.groupby(col, observed=True).agg(пар=("in_top", "size"), top50=("in_top", "mean"), в_пуле=("in_pool", "mean")).round(3).to_string())


релевантных пар: 3342, в top-50: 0.9240, в пуле: 0.9683
потеряно: не попало в пул 0.0317, в пуле но ниже 50-го места 0.0443

           пар  top50  в_пуле
same_loc                     
False      712  0.791   0.886
True      2630  0.960   0.990

               пар  top50  в_пуле
empty_filter                     
False         1735  0.928   0.972
True          1607  0.920   0.965

            пар  top50  в_пуле
text_seen                     
False      2361  0.922   0.966
True        981  0.930   0.973

                                      пар  top50  в_пуле
объявлений корпуса в локации поиска                     
0                                     555  0.816   0.899
1-100                                  51  0.961   0.980
101-1000                              673  0.963   0.984
1001-5000                            1214  0.973   0.993
более 5000                            849  0.892   0.965


Что видно из разбора. Хуже всего идут пары, где выбранное объявление находится не в локации поиска, и запросы из локаций, в которых в корпусе нет объявлений. Идентификаторы локаций между собой не сравнить, поэтому связь берётся из train (матрица переходов) и дополняется расстоянием до центра локации. В крупных локациях с тысячами объявлений ошибок больше на этапе ранкера: слишком много похожих кандидатов. Часть промахов исправить нельзя, потому что пользователи иногда кликают на объявления, слабо связанные с запросом (например, запрос "подруга для прогулок" и объявление фотографа).

По ходу работы нашлись и ошибки в самой постановке эксперимента. Первая версия валидации давала 0.953 против 0.887 на stepik, после перехода на разбиение по объявлениям оценка 0.9185 предсказала 0.9094, а итоговые 0.9241 совпали с stepik 0.9110 в пределах полутора пунктов. Повторяющиеся пары "запрос, объявление" в разметке портили знаменатель Recall, поэтому релевантные объявления считаются множеством.